In [10]:
import pandas as pd
import xarray as xr
import cfgrib
from pathlib import Path
import os

In [11]:
def parse_cfs_filename(filename):
    """
    Parse:
    flxf.01.YYYYMMDDHH.YYYYMM.avrg.grib.grb2
    """

    parts = Path(filename).name.split(".")

    init_time = pd.to_datetime(
        parts[2],
        format="%Y%m%d%H"
    )

    valid_time = pd.to_datetime(
        parts[3],
        format="%Y%m"
    )

    return init_time, valid_time


def read_cfs_variable(file, filter_by_keys, variable_name, output_name=None):

    init_time, valid_time = parse_cfs_filename(file)

    ds = cfgrib.open_dataset(
        file,
        engine="cfgrib",
        filter_by_keys=filter_by_keys,
        decode_timedelta=False,
    )

    # CFS variable name
    da = ds[variable_name]

    # Rename to something cleaner if output_name is provided
    # None is default, I like to keep the original variable name for consistency
    if output_name is not None:
        da = da.rename(output_name)

    da = da.expand_dims(
        cfs_run_time=[init_time],
        valid_time=[valid_time]
    )

    da = da.drop_vars(
        ["time", "step", "heightAboveGround"],
        errors="ignore"    )

    return da

In [12]:
def read_cfs_2m_temperature(file):
    return read_cfs_variable(
        file,
        filter_by_keys={
            "typeOfLevel": "heightAboveGround",
            "level": 2,
        },
        variable_name="avg_2t"
    )

In [13]:
#filename = "/Users/ljob/Desktop/cnbs-predictor/data/cfs/20260501/flxf.01.2026050100.202606.avrg.grib.grb2"
#ds = read_cfs_2m_temperature(filename)
#print(ds)

In [14]:
import glob
from pathlib import Path
from collections import defaultdict


def build_cfs_temperature_archive(
    input_directory,
    output_zarr,
):

    input_directory = Path(input_directory)

    # -------------------------------------------------
    # Precompute all valid forecast months
    # (filename scan only, no GRIB reading)
    # -------------------------------------------------

    all_files = sorted(
        input_directory.rglob("flxf*.grb2")
    )

    all_valid_times = sorted(
        {
            parse_flx_filename(str(file))[1]
            for file in all_files
        }
    )

    first = True

    # -------------------------------------------------
    # Process each initialization directory in order
    # -------------------------------------------------

    init_directories = sorted(
        [
            d for d in input_directory.iterdir()
            if d.is_dir()
        ]
    )

    for init_directory in init_directories:

        files = sorted(
            glob.glob(
                str(init_directory / "flxf*.grb2")
            )
        )

        if len(files) == 0:
            continue

        print(
            f"\nReading in {init_directory.name}"
        )
        print(
            f"    Found {len(files)} files"
        )

        # Group files by initialization time
        grouped = defaultdict(list)

        for file in files:

            init_time, _ = parse_flx_filename(file)

            grouped[init_time].append(file)


        # -------------------------------------------------
        # Process each CFS initialization cycle
        # -------------------------------------------------

        for init_time in sorted(grouped):

            print(
                f"Processing {init_time}"
            )

            monthly = []

            for file in sorted(grouped[init_time]):

                monthly.append(
                    read_cfs_2m_temperature(file)
                )

            run = xr.concat(
                monthly,
                dim="valid_time",
            )

            # Ensure identical valid_time dimension
            run = run.reindex(
                valid_time=all_valid_times
            )

            ds = run.to_dataset()

            if first:

                lat_size = ds.sizes["latitude"]
                lon_size = ds.sizes["longitude"]

                encoding = {
                    "avg_2t": {
                        "chunks": (
                            1,
                            len(all_valid_times),
                            lat_size,
                            lon_size,
                        )
                    },
                    "cfs_run_time": {
                        "units": (
                            "hours since "
                            "1970-01-01 00:00:00"
                        ),
                    },
                    "valid_time": {
                        "units": (
                            "days since "
                            "1970-01-01 00:00:00"
                        ),
                    },
                }

                ds.to_zarr(
                    output_zarr,
                    mode="w",
                    consolidated=True,
                    encoding=encoding,
                    zarr_format=2,
                )

                first = False

            else:

                ds.to_zarr(
                    output_zarr,
                    mode="a",
                    append_dim="cfs_run_time",
                    zarr_format=2,
                )

            del monthly
            del run
            del ds

            print("    written")

            # Remove any cfgrib index files before moving on
            remove_idx_files(init_directory)

    print("Finished.")

import os

def remove_idx_files(directory):
    """
    Remove cfgrib index files (*.idx) from a directory.
    """
    for f in os.listdir(directory):
        if f.endswith(".idx"):
            os.remove(os.path.join(directory, f))

In [15]:
import glob
from collections import defaultdict

def build_cfs_archive(
    input_directory,
    output_zarr,
):

    files = sorted(
        glob.glob(str(Path(input_directory) / "flxf*.grb2"))
    )

    print(f"Found {len(files)} files.")

    # -------------------------------------------------
    # Precompute all valid forecast months
    # -------------------------------------------------

    all_valid_times = sorted(
        {
            parse_cfs_filename(file)[1]
            for file in files
        }
    )

    # Group files by initialization time
    grouped = defaultdict(list)

    for file in files:

        init_time, _ = parse_flx_filename(file)

        grouped[init_time].append(file)

    first = True

    for init_time in sorted(grouped):

        print(f"Processing {init_time}")

        monthly = []

        for file in sorted(grouped[init_time]):

            monthly.append(
                read_cfs_2m_temperature(file)
            )

        run = xr.concat(
            monthly,
            dim="valid_time",
        )

        # Ensure identical valid_time dimension
        run = run.reindex(
            valid_time=all_valid_times
        )

        ds = run.to_dataset()

        if first:

            lat_size = ds.sizes["latitude"]
            lon_size = ds.sizes["longitude"]

            encoding = {
                "avg_2t": {
                    "chunks": (
                        1,
                        len(all_valid_times),
                        lat_size,
                        lon_size,
                    )
                },
                "cfs_run_time": {
                    "units": "hours since 1970-01-01 00:00:00",
                },
                "valid_time": {
                    "units": "days since 1970-01-01 00:00:00",
                },
            }

            ds.to_zarr(
                output_zarr,
                mode="w",
                consolidated=True,
                encoding=encoding,
                zarr_format=2,
            )

            first = False

        else:

            ds.to_zarr(
                output_zarr,
                mode="a",
                append_dim="cfs_run_time",
                zarr_format=2,
            )

        del monthly
        del run
        del ds

        # Cleanup cfgrib index files
        remove_idx_files(input_directory)

        print("    written")

In [16]:
build_cfs_temperature_archive(
    input_directory="/Users/ljob/Desktop/cnbs-predictor/data/cfs",
    output_zarr="/Users/ljob/Desktop/cnbs-predictor/data/zarr/cfs_2m_temperature.zarr",
)


Reading in 20260501
    Found 40 files
Processing 2026-05-01 00:00:00
    written
Processing 2026-05-01 06:00:00
    written
Processing 2026-05-01 12:00:00
    written
Processing 2026-05-01 18:00:00
    written

Reading in 20260502
    Found 40 files
Processing 2026-05-02 00:00:00
    written
Processing 2026-05-02 06:00:00
    written
Processing 2026-05-02 12:00:00
    written
Processing 2026-05-02 18:00:00
    written
Finished.


In [ ]:
import xarray as xr

zarr_path = "/Users/ljob/Downloads/cfs_2m_temperature.zarr"

ds = xr.open_zarr(
    zarr_path,
    consolidated=True,
)


<xarray.Dataset> Size: 117GB
Dimensions:       (cfs_run_time: 2071, valid_time: 194, latitude: 190,
                   longitude: 384)
Coordinates:
  * cfs_run_time  (cfs_run_time) datetime64[ns] 17kB 2011-04-01 ... 2012-10-30
  * valid_time    (valid_time) datetime64[ns] 2kB 2011-04-01 ... 2027-05-01
  * latitude      (latitude) float64 2kB 89.28 88.34 87.4 ... -88.34 -89.28
  * longitude     (longitude) float64 3kB 0.0 0.9375 1.875 ... 357.2 358.1 359.1
Data variables:
    avg_2t        (cfs_run_time, valid_time, latitude, longitude) float32 117GB ...


In [27]:
print(ds["cfs_run_time"][-10:])

<xarray.DataArray 'cfs_run_time' (cfs_run_time: 10)> Size: 80B
array(['2012-10-27T06:00:00.000000000', '2012-10-27T12:00:00.000000000',
       '2012-10-27T18:00:00.000000000', '2012-10-28T00:00:00.000000000',
       '2012-10-28T12:00:00.000000000', '2012-10-28T18:00:00.000000000',
       '2012-10-29T00:00:00.000000000', '2012-10-29T06:00:00.000000000',
       '2012-10-29T18:00:00.000000000', '2012-10-30T00:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * cfs_run_time  (cfs_run_time) datetime64[ns] 80B 2012-10-27T06:00:00 ... 2...


In [18]:
import cfgrib

# Path to GRIB2 file
file = "/Users/ljob/Desktop/Data/flxf06.gdas.200610.grb2"

# Filter for 2-m height above ground variables
filter_by_keys = {
    "typeOfLevel": "heightAboveGround",
    "level": 2,
}

# Open GRIB file
ds = cfgrib.open_dataset(
    file,
    engine="cfgrib",
    filter_by_keys=filter_by_keys,
    decode_timedelta=False,
)

# Print dimensions
print("\n--- Dimensions ---")
for dim, size in ds.sizes.items():
    print(f"{dim}: {size}")

# Print coordinates
print("\n--- Coordinates ---")
for coord in ds.coords:
    print(f"{coord}: {ds[coord].dims} {ds[coord].shape}")

# Print variables
print("\n--- Variables ---")
for var in ds.data_vars:
    print(f"{var}:")
    print(f"  Dimensions: {ds[var].dims}")
    print(f"  Shape: {ds[var].shape}")
    print(f"  Units: {ds[var].attrs.get('units', 'unknown')}")
    print(f"  Long name: {ds[var].attrs.get('long_name', 'unknown')}")

# Optional: print full dataset summary
print("\n--- Full Dataset ---")
print(ds)


--- Dimensions ---
latitude: 576
longitude: 1152

--- Coordinates ---
time: () ()
step: () ()
heightAboveGround: () ()
latitude: ('latitude',) (576,)
longitude: ('longitude',) (1152,)
valid_time: () ()

--- Variables ---
t2m:
  Dimensions: ('latitude', 'longitude')
  Shape: (576, 1152)
  Units: K
  Long name: 2 metre temperature
sh2:
  Dimensions: ('latitude', 'longitude')
  Shape: (576, 1152)
  Units: kg kg**-1
  Long name: 2 metre specific humidity
tmax:
  Dimensions: ('latitude', 'longitude')
  Shape: (576, 1152)
  Units: K
  Long name: Maximum temperature
tmin:
  Dimensions: ('latitude', 'longitude')
  Shape: (576, 1152)
  Units: K
  Long name: Minimum temperature
qmax:
  Dimensions: ('latitude', 'longitude')
  Shape: (576, 1152)
  Units: kg kg**-1
  Long name: Maximum specific humidity at 2m
qmin:
  Dimensions: ('latitude', 'longitude')
  Shape: (576, 1152)
  Units: kg kg**-1
  Long name: Minimum specific humidity at 2m

--- Full Dataset ---
<xarray.Dataset> Size: 16MB
Dimensions